# Module 03: Searching & Sorting Operations in NumPy

Welcome to **Module 03** of the NumPy Reference Repository. This module focuses on high-performance vector searching, sorting algorithms, conditional indexing, and linear-time selection mechanics.

---

### Module Objectives & Scope
1. **Sorting Mechanics:** Distinguish between out-of-place (`np.sort`) and in-place (`arr.sort`) mutations, alongside index-based sorting (`np.argsort`).
2. **Conditional Searching:** Master vectorized `if-else` branching (`np.where`), set membership checks (`np.isin`), and non-zero index extraction (`np.nonzero`).
3. **Logarithmic Lookup ($O(\log N)$):** Utilize binary search methods (`np.searchsorted`) and understand boundary conditions (`side='left'` vs `side='right'`).
4. **Linear-Time Partial Sorting ($O(N)$):** Leverage `np.partition` and `np.argpartition` for optimal Top-$K$ element extraction without full array sorts.
5. **Multi-Key Sorting:** Implement stable lexicographical sorting across multiple feature columns using `np.lexsort`.

---

### Key Concepts & Memory Logic: Sorting Operation

- **Array Sorting (`np.sort` vs. `.sort()`):**
  - **`np.sort(a, axis=-1, kind=None)` (Function / Out-of-place):** Performs an out-of-place sort. It leaves the original array untouched and allocates a brand-new contiguous memory block (**Deep Copy**, $O(N)$ space complexity).
  - **`a.sort(axis=-1, kind=None)` (Method / In-place):** Sorts the array directly in-place without creating a copy ($O(1)$ auxiliary memory complexity). Returns `None` and mutates the original buffer.
- **Index-based Sorting (`np.argsort`):**
  - Returns an array of indices that would sort the input array rather than sorting the values directly.
  - Critical for relational data alignment (e.g., sorting student names based on their exam scores).
- **Sorting Algorithms & Stability:**
  - Default algorithm is Quicksort (`kind='quicksort'`).
  - For guaranteed preservation of duplicate element ordering, use stable algorithms (`kind='stable'` or `kind='mergesort'`).
  - Time complexity is $O(N \log N)$ on average for 1D arrays.
- **Axis-wise Multi-dimensional Sorting:**
  - **`axis=1` (Row-wise):** Sorts values independently across each individual row.
  - **`axis=0` (Column-wise):** Sorts values independently down each individual column.

In [1]:
import numpy as np

In [2]:
# 1. 1D Array Sorting: Function vs Method
arr_1d = np.array([32, 12, 5, 89, 23, 44])

# Out-of-place sort (Returns a new sorted copy)
sorted_copy = np.sort(arr_1d)
print("Original Array (Unchanged):", arr_1d)
print("Sorted Copy (np.sort):     ", sorted_copy)

# Memory ownership check: np.sort returns a Copy, not a View
print("Is sorted_copy a View?:", sorted_copy.base is arr_1d)  # Outputs False

# In-place sort (Modifies the original array directly)
arr_1d_copy = arr_1d.copy()
arr_1d_copy.sort()
print("In-place Sorted Array (.sort()):", arr_1d_copy)


# 2. Index-based Sorting (np.argsort)
scores = np.array([85, 92, 45, 78])
student_ids = np.array(["ID_A", "ID_B", "ID_C", "ID_D"])

# Get the indices that would sort the scores array
sorted_indices = np.argsort(scores)
print("\nScores:        ", scores)
print("Sorted Indices:", sorted_indices)  # Outputs: [2 3 0 1] -> 45, 78, 85, 92

# Use argsort indices to align and sort associated student data
print("Students Sorted by Score:", student_ids[sorted_indices])


# 3. 2D Array Sorting Along Axes
matrix = np.array([
    [12, 5, 29],
    [4, 88, 1],
    [33, 2, 15]
])

# Axis 1 (Row-wise sorting: Sort elements within each row)
sorted_rows = np.sort(matrix, axis=1)
print("\n2D Matrix Sorted Row-wise (axis=1):")
print(sorted_rows)

# Axis 0 (Column-wise sorting: Sort elements within each column)
sorted_cols = np.sort(matrix, axis=0)
print("\n2D Matrix Sorted Column-wise (axis=0):")
print(sorted_cols)

Original Array (Unchanged): [32 12  5 89 23 44]
Sorted Copy (np.sort):      [ 5 12 23 32 44 89]
Is sorted_copy a View?: False
In-place Sorted Array (.sort()): [ 5 12 23 32 44 89]

Scores:         [85 92 45 78]
Sorted Indices: [2 3 0 1]
Students Sorted by Score: ['ID_C' 'ID_D' 'ID_A' 'ID_B']

2D Matrix Sorted Row-wise (axis=1):
[[ 5 12 29]
 [ 1  4 88]
 [ 2 15 33]]

2D Matrix Sorted Column-wise (axis=0):
[[ 4  2  1]
 [12  5 15]
 [33 88 29]]


> 📌 **Takeaways & Rules of Thumb:**
> 1. **Data Protection:** Use `np.sort()` when you need to preserve the raw dataset and create a new sorted array.
> 2. **Memory Efficiency:** Use `arr.sort()` when working with large arrays to save memory through in-place modification ($O(1)$ extra space).
> 3. **Relational Data Alignment:** Use `np.argsort()` when sorting multiple arrays/columns based on a single key metric.

### Key Concepts & Memory Logic: Searching Operations

- **Conditional Selection (`np.where`):**
  - Acts as a vectorized `if-else` construct.
  - When given only a condition `np.where(condition)`, it returns a tuple of index arrays where the condition is `True`.
  - When given values `np.where(condition, x, y)`, it evaluates element-wise: yields $x$ if `True`, else $y$.
- **Extrema Indexing (`np.argmin` / `np.argmax`):**
  - Finds the flat index or axis-wise index of the minimum or maximum element.
  - Extremely fast ($O(N)$) for feature engineering and model evaluations.
- **Binary Search on Sorted Arrays (`np.searchsorted`):**
  - Uses Binary Search ($O(\log N)$ time complexity) to find indices where elements should be inserted to maintain order.
  - Requires input array to be pre-sorted.
- **Non-Zero Element Selection (`np.nonzero` / `np.flatnonzero`):**
  - Extracts indices of elements that are non-zero or satisfy boolean evaluation.

In [3]:

# 1. Vectorized Conditional Selection: np.where
data = np.array([12, 45, 67, 23, 89, 34, 10])

# Scenario A: Get indices matching condition (> 30)
indices_gt_30 = np.where(data > 30)
print("Indices where data > 30:", indices_gt_30[0])
print("Values where data > 30: ", data[indices_gt_30])

# Scenario B: Vectorized ternary operator np.where(condition, x, y)
# Replace values > 30 with 999, others with -1
transformed_data = np.where(data > 30, 999, -1)
print("Transformed Array:      ", transformed_data)

# Scenario C: Get indices of even numbers and odd numbers
result_even=np.where(data%2==0)
print("Indices of even numbers:", result_even)
result_odd=np.where(data%2==1)
print("Indices of odd numbers:", result_odd)


Indices where data > 30: [1 2 4 5]
Values where data > 30:  [45 67 89 34]
Transformed Array:       [ -1 999 999  -1 999 999  -1]
Indices of even numbers: (array([0, 5, 6]),)
Indices of odd numbers: (array([1, 2, 3, 4]),)


> 📌 **Takeaways & Rules of Thumb:**
> 1. **`np.where` Efficiency:** Use single-argument `np.where(cond)` for conditional indexing, and 3-argument `np.where(cond, x, y)` for element-wise conditional substitution.
> 2. **Binary Search ($O(\log N)$):** Always prefer `np.searchsorted()` over linear scanning when searching inside an already sorted array.
> 3. **Memory Note:** `np.where` creates a new array of indices or values, making it an allocation-heavy $O(N)$ operation.

### Key Concepts & Memory Logic: Set Membership (`np.isin`)

- **Vectorized Set Membership (`np.isin`):**
  - Evaluates whether each element of an input array is present in a target list or array.
  - Returns a **Boolean array** of the exact same shape as the input array.
  - Highly efficient vectorized equivalent of Python's `x in sequence` or list comprehension logic.
- **Invert Behavior (`invert=True`):**
  - Reverses the logical outcome (acts as "NOT IN"). Returns `True` for elements **not** present in the target values.
- **Memory & Allocation Behavior:**
  - Allocates a brand-new boolean mask array ($O(N)$ space complexity).
  - Does **not** modify or mutate the original input array.

In [4]:

# 1. Basic Set Membership Check
dataset = np.array([10, 25, 30, 45, 50, 60, 75])
target_values = [25, 50, 90]  # Values to search for

# Returns a boolean mask matching dataset's shape
mask = np.isin(dataset, target_values)

print("Dataset:      ", dataset)
print("Is in Targets:", mask)
print("Matched Values:", dataset[mask])


# 2. Inverted Membership (NOT IN filtering)
# Filter out elements that are in target_values
not_in_mask = np.isin(dataset, target_values, invert=True)

print("\nFiltered Dataset (Excluding Targets):", dataset[not_in_mask])


# 3. Multi-dimensional Support
matrix = np.array([
    [1, 2, 3],
    [4, 5, 6]
])
allowed_set = [2, 5]

# Preserves 2D shape in output boolean mask
matrix_mask = np.isin(matrix, allowed_set)
print("\n2D Matrix Mask Shape:", matrix_mask.shape)
print("2D Matrix Mask:\n", matrix_mask)

Dataset:       [10 25 30 45 50 60 75]
Is in Targets: [False  True False False  True False False]
Matched Values: [25 50]

Filtered Dataset (Excluding Targets): [10 30 45 60 75]

2D Matrix Mask Shape: (2, 3)
2D Matrix Mask:
 [[False  True False]
 [False  True False]]


### Key Concepts: Non-Zero Element Selection (`np.nonzero` & `np.flatnonzero`)

- **`np.nonzero(a)`:**
  - Returns a tuple of arrays containing the indices of elements that are **non-zero** (or `True` in boolean masks).
  - The length of the tuple equals `a.ndim` (1 array per dimension).
- **`np.flatnonzero(a)`:**
  - Flattens the array in memory view first and returns a 1D array of non-zero indices.
  - Extremely useful for quick 1D index retrieval on flattened arrays.
- **Memory & Behavior:**
  - Creates a new integer array of indices ($O(N)$ allocation).
  - Equivalent to `np.where(a != 0)`.

In [5]:

# 1. 1D Array Non-Zero Indexing
sparse_1d = np.array([0, 12, 0, 0, 45, 0, 89])

# Get indices of non-zero elements
non_zero_indices = np.nonzero(sparse_1d)

print("1D Sparse Array:    ", sparse_1d)
print("Non-zero Indices:   ", non_zero_indices[0])  # Outputs: [1 4 6]
print("Non-zero Values:    ", sparse_1d[non_zero_indices])


# 2. 2D Matrix Non-Zero Indexing
matrix_2d = np.array([
    [0, 5, 0],
    [7, 0, 0],
    [0, 0, 9]
])

row_idx, col_idx = np.nonzero(matrix_2d)

print("\n2D Matrix:\n", matrix_2d)
print("Row Indices:   ", row_idx)  # Outputs: [0, 1, 2]
print("Column Indices:", col_idx)  # Outputs: [1, 0, 2]

# Zip indices to show exact coordinates (row, col)
coordinates = list(zip(row_idx, col_idx))
print("Coordinates (row, col) of non-zero elements:", coordinates)


# 3. Flattened Non-Zero Indexing: np.flatnonzero
flat_idx = np.flatnonzero(matrix_2d)
print("\nFlat Non-zero Indices (Linear index):", flat_idx)  # Outputs: [1, 3, 8]

1D Sparse Array:     [ 0 12  0  0 45  0 89]
Non-zero Indices:    [1 4 6]
Non-zero Values:     [12 45 89]

2D Matrix:
 [[0 5 0]
 [7 0 0]
 [0 0 9]]
Row Indices:    [0 1 2]
Column Indices: [1 0 2]
Coordinates (row, col) of non-zero elements: [(np.int64(0), np.int64(1)), (np.int64(1), np.int64(0)), (np.int64(2), np.int64(2))]

Flat Non-zero Indices (Linear index): [1 3 8]


### Key Concepts & Memory Logic: Binary Search (`np.searchsorted`)

- **Logarithmic Time Complexity ($O(\log N)$):**
  - Performs a **Binary Search** to find the insertion indices that maintain sorted order.
  - Requires the input base array to be **pre-sorted** in ascending order.
- **The `side` Parameter (`'left'` vs `'right'`):**
  - **`side='left'` (Default):** Returns the index of the **first** suitable insertion point. For duplicates, it inserts to the **left** of existing values.
  - **`side='right'`:** Returns the index of the **last** suitable insertion point. For duplicates, it inserts to the **right** of existing values.
- **Vectorized Searching:**
  - Can evaluate multiple target values simultaneously in a single call.
- **Memory & Allocation:**
  - Allocates a new array of integer indices matching the shape of the target array ($O(K)$ space complexity, where $K$ is the number of target values).

In [6]:

# Base array MUST be sorted
sorted_arr = np.array([4, 7, 11, 13, 18, 19])

# 1. Single Value Insertion Search
target_val = 13

# Side behavior on duplicates/existing elements
idx_left = np.searchsorted(sorted_arr, target_val, side='left')
idx_right = np.searchsorted(sorted_arr, target_val, side='right')

print("Sorted Base Array:", sorted_arr)
print(f"Insertion Index for {target_val} (side='left'): ", idx_left)   # Outputs: 3
print(f"Insertion Index for {target_val} (side='right'):", idx_right)  # Outputs: 4


# 2. Vectorized Binary Search (Multiple Targets)
targets = np.array([2, 10, 13, 25])
insert_positions = np.searchsorted(sorted_arr, targets)

print("\nTarget Values:     ", targets)
print("Insertion Indices: ", insert_positions)  # Outputs: [0, 2, 3, 6]


# 3. Practical Use-Case: Categorizing / Binning Continuous Data
# Define bin edges: [< 10: Low], [10-18: Medium], [> 18: High]
bins = np.array([10, 18])
scores = np.array([5, 12, 15, 22, 8, 18])

# Assign bin indices automatically in O(log N) time
binned_indices = np.searchsorted(bins, scores)
print("\nScores:        ", scores)
print("Bin Bucket Idx:", binned_indices)

Sorted Base Array: [ 4  7 11 13 18 19]
Insertion Index for 13 (side='left'):  3
Insertion Index for 13 (side='right'): 4

Target Values:      [ 2 10 13 25]
Insertion Indices:  [0 2 3 6]

Scores:         [ 5 12 15 22  8 18]
Bin Bucket Idx: [0 1 1 2 0 1]


In [7]:
# 2. Extrema Searching: np.argmin & np.argmax
matrix = np.array([
    [10, 50, 30],
    [70, 20, 90],
    [40, 80, 60]
])

print("\nMatrix:\n", matrix)
print("Flat Argmax (Index of overall max value):", np.argmax(matrix))  # Index 5 (Value: 90)
print("Column-wise Argmax (axis=0):              ", np.argmax(matrix, axis=0))
print("Row-wise Argmax (axis=1):                 ", np.argmax(matrix, axis=1))


# 3. Optimized Binary Search: np.searchsorted
sorted_arr = np.array([10, 20, 30, 40, 50, 60])
target_values = np.array([25, 5, 45])

# Find insertion indices to maintain sorted order
insert_positions = np.searchsorted(sorted_arr, target_values)
print("\nSorted Base Array:    ", sorted_arr)
print("Target Values:        ", target_values)
print("Insertion Positions:  ", insert_positions)  # Outputs: [2, 0, 4]


# 4. Non-Zero Indexing: np.nonzero
sparse_arr = np.array([0, 5, 0, 0, 12, 0, 8])
non_zero_idx = np.nonzero(sparse_arr)
print("\nSparse Array:            ", sparse_arr)
print("Non-zero Element Indices:", non_zero_idx[0])


Matrix:
 [[10 50 30]
 [70 20 90]
 [40 80 60]]
Flat Argmax (Index of overall max value): 5
Column-wise Argmax (axis=0):               [1 2 1]
Row-wise Argmax (axis=1):                  [1 2 1]

Sorted Base Array:     [10 20 30 40 50 60]
Target Values:         [25  5 45]
Insertion Positions:   [2 0 4]

Sparse Array:             [ 0  5  0  0 12  0  8]
Non-zero Element Indices: [1 4 6]


## Summary: Memory Architecture & Complexity for Module 03

The following table summarizes the time and space complexity, memory ownership behaviors, and optimal use cases for all Searching & Sorting operations covered in Module 03.

| Function / Method | Primary Functionality | Average Time Complexity | Auxiliary Space Complexity | Memory Ownership Output |
| :--- | :--- | :--- | :--- | :--- |
| **`np.sort(a)`** | Returns a sorted copy of an array | $O(N \log N)$ | $O(N)$ | **New Array (Copy)** (`.base is None`) |
| **`a.sort()`** | Sorts array directly in-place | $O(N \log N)$ | $O(1)$ | **In-Place Mutation** (Returns `None`) |
| **`np.argsort(a)`** | Returns indices that would sort an array | $O(N \log N)$ | $O(N)$ | **New Array (Copy)** of integer indices |
| **`np.where(cond)`** | Returns index tuple where condition is `True` | $O(N)$ | $O(N)$ | **New Tuple of Arrays** |
| **`np.where(cond, x, y)`** | Element-wise ternary substitution | $O(N)$ | $O(N)$ | **New Array (Copy)** |
| **`np.searchsorted(a, v)`**| Binary search for insertion indices | $O(K \log N)$ | $O(K)$ | **New Array (Copy)** of integer indices |
| **`np.isin(element, test)`**| Vectorized set membership evaluation | $O(N \times M)$ | $O(N)$ | **New Boolean Mask Array** |
| **`np.nonzero(a)`** | Returns tuple of non-zero element indices | $O(N)$ | $O(N)$ | **New Tuple of Arrays** |
| **`np.flatnonzero(a)`** | Returns 1D non-zero indices on flattened view | $O(N)$ | $O(N)$ | **New 1D Array (Copy)** |
| **`np.lexsort(keys)`** | Indirect stable multi-key sort | $O(N \log N)$ | $O(N)$ | **New Array (Copy)** of integer indices |
| **`np.partition(a, kth)`**| Partial sorting around the $k$-th element | $O(N)$ | $O(N)$ | **New Array (Copy)** |

> 📌 **Module 03 Rules of Thumb & Engineering Best Practices:**
>
> 1. **In-Place vs. Out-of-Place Sorting:** Use `a.sort()` for large memory-constrained arrays to modify in-place ($O(1)$ space). Use `np.sort(a)` when raw data immutability is required.
> 2. **Relational Alignment via `np.argsort()`:** Never sort multi-column or relational arrays independently. Use `np.argsort()` on the key feature and index into all related arrays.
> 3. **Binary Search Mechanics (`np.searchsorted`):** Requires a pre-sorted base array ($O(\log N)$). Remember that `side='left'` inserts before existing duplicate values, while `side='right'` inserts after them.
> 4. **Boolean Indexing (`arr[mask]`):** Vectorized set membership via `np.isin()` and condition evaluations via `np.where()` produce boolean masks. Indexing with `arr[mask]` extracts values where the mask evaluates to `True`.
> 5. **Non-Zero Indexing (`np.nonzero` vs `np.flatnonzero`):** Use `np.nonzero()` to preserve original multidimensional coordinate tuples, or `np.flatnonzero()` when 1D linear indices are preferred.
> 6. **Top-K Partial Selection ($O(N)$ vs $O(N \log N)$):** Avoid full sorts (`np.sort()`) when extracting top-$K$ or bottom-$K$ items. Use `np.partition()` or `np.argpartition()` for linear-time performance.